In [8]:
from utils import write_yaml, read_yaml
import pandas as pd  # Fixed: was "panda"
import json
file_path = "../GovData/policecalls/policecalls.csv"
df = pd.read_csv(file_path)

In [36]:
# select_dtype() Filtra colunas por tipo de dados
# kurtosis é um tipo de 
df.select_dtypes(include=['number']).corr().to_dict()

{'lat': {'lat': 1.0, 'lng': -0.13397329234999794},
 'lng': {'lat': -0.13397329234999794, 'lng': 1.0}}

In [ ]:
def pre_analysis(path: str, file_name: str):
    data = pd.read_csv(path+file_name)
    metadata = {
    "success": True,
    # Descreve cada coluna identificando valores de mínimo, máximo e desvio padrão (numeros), 
    # tipos mais frequentes, frequencia dos tipos mais frequentes e tipos unicos (texto)
    "describe": data.describe().to_dict(),
    "kurtosis": data.select_dtypes(include=['number']).kurtosis().to_dict(),
    "skewness": data.select_dtypes(include=['number']).skew().to_dict(),
    "correlation_matrix": data.select_dtypes(include=['number']).corr().to_dict(),
    "shape": list(data.shape), # dimensões da matriz
    "columns": data.columns.tolist(), # colunas
    "dtypes": data.dtypes.astype(str).to_dict(), # tipos de cada coluna
    "sample_data": data.head().to_dict('records'), # Cinco primeiros valores
    "null_counts": data.isnull().sum().to_dict(), # contagem de células vazias
    "memory_usage": float(data.memory_usage(deep=True).sum()/(1024**2)) # Quantidade de memoria RAM em Mb
    }
    
    write_yaml(origin_path=data, destiny_path=path+"analysis.yaml", data=metadata, overwrite=True)

    return metadata

pre_analysis(path="../GovData/policecalls/", file_name="policecalls.csv")

In [40]:
metadata = read_yaml(path="../GovData/policecalls/analysis.yaml")
metadata

{'columns': ['date', 'type', 'lat', 'lng'],
 'correlation_matrix': {'lat': {'lat': 1.0, 'lng': -0.13397329234999794},
  'lng': {'lat': -0.13397329234999794, 'lng': 1.0}},
 'describe': {'lat': {'25%': -3.7873,
   '50%': -3.75552,
   '75%': -3.73143,
   'count': 135760.0,
   'max': -2.17874,
   'mean': -3.763318391573365,
   'min': -85.1819,
   'std': 0.3845193151681059},
  'lng': {'25%': -38.5815,
   '50%': -38.5526,
   '75%': -38.513,
   'count': 135760.0,
   'max': 2.45771,
   'mean': -38.546232200869184,
   'min': -38.723,
   'std': 0.120005299485865}},
 'dtypes': {'date': 'object',
  'lat': 'float64',
  'lng': 'float64',
  'type': 'object'},
 'kurtosis': {'lat': 44418.51120472207, 'lng': 100420.27842391877},
 'memory_usage': 20.324875831604004,
 'null_counts': {'date': 0, 'lat': 0, 'lng': 0, 'type': 0},
 'sample_data': [{'date': '2005-01-01',
   'lat': -3.73784,
   'lng': -38.5554,
   'type': 'PROPERTY CRIMES'},
  {'date': '2005-01-01',
   'lat': -3.83914,
   'lng': -38.5606,
   'ty

In [16]:
import pandas as pd  # Fixed: was "panda"
import json
from typing import Dict, Any, List, Optional
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.graph import MessagesState

@tool
def read_csv_file(file_path: str, delimiter: str = ';', encoding: str = 'utf-8') -> Dict[str, Any]:
    """Read and analyze a CSV file, returning basic information about its structure and content."""
    try:
        df = pd.read_csv(file_path, delimiter=delimiter, encoding=encoding)

        return {
            "success": True,
            "shape": df.shape,
            "columns": df.columns.tolist(),
            "dtypes": df.dtypes.astype(str).to_dict(),
            "sample_data": df.head().to_dict('records'),
            "null_counts": df.isnull().sum().to_dict(),
            "memory_usage": df.memory_usage(deep=True).sum()
        }
    except Exception as e:
        return {
            "success": False,
            "error": str(e)
        }
    
@tool
def analyze_csv_data(file_path: str, analysis_type: str = "basic") -> Dict[str, Any]:
    """Perform statistical analysis on CSV data with different analysis types (basic, detailed, correlation)."""
    try:
        df = pd.read_csv(file_path, delimiter=';')  # Match your CSV format
        if analysis_type == "basic":
            return {
                "success": True,
                "describe": df.describe().to_dict(),
                "value_counts": {col: df[col].value_counts().head().to_dict()
                                for col in df.select_dtypes(include=['object']).columns}
            }
        elif analysis_type == "detailed":
            return {
                "success": True,
                "describe": df.describe(include='all').to_dict(),
                "skewness": df.select_dtypes(include=['number']).skew().to_dict(),
                "kurtosis": df.select_dtypes(include=['number']).kurtosis().to_dict()
            }
        elif analysis_type == "correlation":
            corr_matrix = df.select_dtypes(include=['number']).corr()
            return {
                "success": True,
                "correlation_matrix": corr_matrix.to_dict()
            }
    except Exception as e:
        return {
            "success": False,
            "error": str(e)
        }

@tool
def query_csv_data(file_path: str, query: str) -> Dict[str, Any]:
    """Execute pandas query operations on CSV data to filter and analyze specific subsets."""
    try:
        csv = pd.read_csv(file_path, delimiter=';')  # Match your CSV format
        result = csv.query(query)
        
        return {
            "success": True,
            "result_shape": result.shape,
            "data": result.to_dict('records')
        }
    except Exception as e:
        return {
            "success": False,
            "error": str(e)
        }

In [ ]:
class CSVAnalyzerState(MessagesState):
    csv_path: str
    csv_data: Optional[Dict[str, Any]] = None
    analysis_results: Optional[Dict[str, Any]] = None  # Fixed: was "analysis_result"
    user_query: str = ""

def load_csv_node(state: CSVAnalyzerState) -> CSVAnalyzerState:
    
    result = read_csv_file.invoke({"file_path": state["csv_path"]})
    return{
        "csv_data": result,
        "messages": [f"CSV loaded: {result.get('shape', 'Error')} rows x columns"]
    }

def analyze_csv_node(state: CSVAnalyzerState) -> CSVAnalyzerState:
    if not state.get("csv_data",{}).get("success"):
        return {"messages": ["Não é possível analisar - Carregamento do CSV falhou."]}
    
  
    analysis = analyze_csv_data.invoke({
        "file_path": state["csv_path"], 
        "analysis_type": "basic"
    })
    return {
        "analysis_results": analysis,
        "messages": [f"Análise concluída: {len(analysis.get('describe', {}))} colunas analisadas."]
    }

def query_csv_node(state: CSVAnalyzerState):
    if state.get("user_query"):
   
        result = query_csv_data.invoke({
            "file_path": state["csv_path"], 
            "query": state["user_query"]
        })
        return {"messages": [f"Consulta executada: {result.get('result_shape', 'Error')}"]}
    return {"messages": ["Nenhuma consulta fornecida."]}


In [22]:
def criar_csv_analyzer_graph() :
    workflow = StateGraph(CSVAnalyzerState)

    workflow.add_node("Carregar CSV", load_csv_node)
    workflow.add_node("Analisar CSV", analyze_csv_node)
    workflow.add_node("Consultar CSV", query_csv_node)

    workflow.set_entry_point("Carregar CSV")
    workflow.add_edge("Carregar CSV", "Analisar CSV")
    workflow.add_edge("Analisar CSV", "Consultar CSV")
    workflow.add_edge("Consultar CSV", END)
    
    return workflow.compile()

def executar_analise_csv(csv_path: str, user_query: str = ""):
    graph = criar_csv_analyzer_graph()
    initial_state = {
        "csv_path": csv_path,
        "user_query": user_query,
        "messages": []
    }
    final_state = graph.invoke(initial_state)  # Fixed: was "graph.run"
    return final_state
print(executar_analise_csv("c:\\Users\\mmari\\Downloads\\2023-04-19_Empresas_Funcoes_OEA.csv", "coluna1 > 100"))


{'messages': [HumanMessage(content='CSV loaded: (84, 10) rows x columns', additional_kwargs={}, response_metadata={}, id='9a18a056-ccae-4382-a3c2-8af72a38d28c'), HumanMessage(content='Análise concluída: 9 colunas analisadas.', additional_kwargs={}, response_metadata={}, id='123b929b-6fab-4d34-8908-190130661004'), HumanMessage(content='Consulta executada: Error', additional_kwargs={}, response_metadata={}, id='3d2d3d20-4255-44a4-a4b6-eb3cb3faf15b')], 'csv_path': 'c:\\Users\\mmari\\Downloads\\2023-04-19_Empresas_Funcoes_OEA.csv', 'csv_data': {'success': True, 'shape': (84, 10), 'columns': ['Mes', 'Depositario (OEA-S)', 'Importador / Exportador (OEA-C1)', 'Importador / Exportador (OEA-C)', 'Importador / Exportador (OEA-Integrado Secex)', 'Importador / Exportador (OEA-S)', 'Operador Aeroportuario (OEA-S)', 'Operador Portuario (OEA-S)', 'Redex (OEA-S)', 'Transportador (OEA-S)'], 'dtypes': {'Mes': 'object', 'Depositario (OEA-S)': 'int64', 'Importador / Exportador (OEA-C1)': 'float64', 'Impor

In [21]:
csv_path = r"c:\Users\mmari\Downloads\2023-04-19_Empresas_Funcoes_OEA.csv"

if __name__ == "__main__":
    executar_analise_csv("csv_path", "")
